# 冻结多时点预测进入完整Stage1的Tiered检验

复用已完成的分钟模型，不重新训练。开盘动作保留现役模型；盘中在原风险/恢复权限内比较HOLD、基础、降杠杆与恢复目标。固定首小时主候选及数值分钟中性对照，其他两组只作诊断。各账户统一15分钟延迟close口径，与生产09:30开盘区别明确。

In [1]:
import os,sys,json
from pathlib import Path
root=Path.cwd()
while not (root/"SPEC.md").is_file():root=root.parent
sys.path.insert(0,str(root));os.environ["MLFLOW_ALLOW_FILE_STORE"]="true"
card=root/"research/stage1_multiclock_tiered_20261013"
plan=json.loads((card/"plan.json").read_text())
from src.workflow.stage1_multiclock import prepare_multiclock,run_multiclock
prepared=prepare_multiclock(root=root,card=card,plan=plan)
summary=run_multiclock(root=root,card=card,plan=plan,prepared=prepared)
summary

{'experiment': 'stage1_multiclock_tiered_20261013',
 'plan_sha256': 'c802759212306f7e9ad8820be96e8e5ef07d8aa167eed77658d9080e89f48b30',
 'decision': 'not_promoted_multiclock_full_stage1_gates_failed',
 'objective_achieved': False,
 'goal_remains_active': True,
 'new_model_fits': 0,
 'native_cases': 18,
 'gates': {'net_soxx_excess_all_costs': False,
  'monitored_drawdown_within_30': False,
  'cvar_no_worse_than_soxx': True,
  'minute_increment_all_costs': False,
  'rolling_win_70pct': False,
  'positive_years_at_least_three_all_costs': False,
  'paired_bootstrap_excess_and_increment': False,
  'qualified_independent_forward': False},
 'primary_family': 'first_hour',
 'family_reselection': False,
 'start': '2023-01-04',
 'end': '2026-09-18',
 'as_of': '2026-10-09',
 'provider_uri': '/Users/rziiiii/dev/workspace/data/qlib/stage1_multiclock_tiered_20261013/execution',
 'recorder_id': 'b6fa35570534402fb8721a81128e0679',
 'experiment_id': '540542923399766386',
 'recorder_uri': '/Users/rziiii

In [2]:
import pandas as pd
from IPython.display import Markdown,display
display(pd.read_csv(card/"backtest_benchmark_comparison.csv"))
display(Markdown((card/"conclusion.md").read_text()))

,family,slippage_bps,sessions,return_pct,cagr_pct,max_drawdown_pct,cvar5_pct,sharpe,benchmark_return_pct,relative_wealth,benchmark_cvar5_pct,monitored_drawdown_pct,broker_fees_usd,slippage_usd,orders,intraday_actions,mandatory_repairs,average_cash_pct,maximum_soxl_weight_pct
0,current,0,930,67.197770,14.944517,-29.317679,-4.909950,0.605390,365.657934,0.359057,-5.405783,-30.376906,1202.958244,0.000000,1116,0,0,54.641452,32.215164
1,neutral_first_hour,0,930,201.150454,34.813968,-28.512442,-4.968946,1.051383,365.657934,0.646720,-5.405783,-28.670423,4359.913406,0.000000,3507,1583,1063,44.306469,31.339255
2,first_hour,0,930,177.002178,31.794920,-30.662551,-4.696718,1.007534,365.657934,0.594862,-5.405783,-31.816806,6726.834921,0.000000,3550,1522,970,46.212799,31.341342
3,full_day,0,930,103.844910,21.285892,-31.819579,-4.631968,0.773212,365.657934,0.437757,-5.405783,-33.588679,9964.002487,0.000000,4501,1990,993,46.055543,31.342807
4,full_day_context,0,930,113.934501,22.884031,-30.498918,-4.514364,0.827554,365.657934,0.459424,-5.405783,-32.070190,11668.333027,0.000000,4795,2128,1000,45.096775,31.335492
5,soxx,0,930,365.657934,51.713633,-41.325190,-5.405783,1.288054,365.657934,1.000000,-5.405783,-43.402322,1.904383,0.000000,1,0,0,0.101499,0.000000
6,current,2,930,32.165756,7.849796,-31.286148,-4.522113,0.408056,365.637973,0.283838,-5.406213,-32.309683,1038.999035,1389.382443,994,0,0,58.082603,32.219850
7,neutral_first_hour,2,930,228.107797,37.982446,-28.894487,-5.026243,1.099814,365.637973,0.704641,-5.406213,-29.249116,3055.413057,6348.764003,3055,1375,1061,45.846690,31.339954
8,first_hour,2,930,172.026681,31.149216,-31.369893,-4.805792,0.985337,365.637973,0.584202,-5.406213,-32.513001,4745.277522,9213.430186,3154,1351,996,47.163460,31.333880
9,full_day,2,930,120.477298,23.891221,-31.316403,-4.613454,0.845842,365.637973,0.473495,-5.406213,-32.372534,8539.470195,15066.681107,4015,1744,971,45.821452,31.340959


# 多时点完整Stage1费用检验

冻结分钟模型与预测复用，新增拟合0次；18个实际Tiered/0-2-5bp账户通过Qlib逐笔费用、现金、股数及净值对账。原首小时主候选保留，其他两组不事后重选。

|模型|滑点bp|净收益%|15分钟监测回撤%|相对SOXX财富|
|---|---:|---:|---:|---:|
|current|0|67.20|-30.38|0.3591|
|neutral_first_hour|0|201.15|-28.67|0.6467|
|first_hour|0|177.00|-31.82|0.5949|
|full_day|0|103.84|-33.59|0.4378|
|full_day_context|0|113.93|-32.07|0.4594|
|soxx|0|365.66|-43.40|1.0000|
|current|2|32.17|-32.31|0.2838|
|neutral_first_hour|2|228.11|-29.25|0.7046|
|first_hour|2|172.03|-32.51|0.5842|
|full_day|2|120.48|-32.37|0.4735|
|full_day_context|2|167.68|-30.58|0.5749|
|soxx|2|365.64|-43.41|1.0000|
|current|5|113.00|-29.13|0.4575|
|neutral_first_hour|5|196.31|-30.70|0.6364|
|first_hour|5|188.25|-30.81|0.6191|
|full_day|5|90.29|-32.33|0.4087|
|full_day_context|5|63.35|-33.23|0.3508|
|soxx|5|365.61|-43.41|1.0000|

门槛：{"net_soxx_excess_all_costs": false, "monitored_drawdown_within_30": false, "cvar_no_worse_than_soxx": true, "minute_increment_all_costs": false, "rolling_win_70pct": false, "positive_years_at_least_three_all_costs": false, "paired_bootstrap_excess_and_increment": false, "qualified_independent_forward": false}。原开盘动作、日级风险状态与恢复许可保留，盘中经济选择及中性对照共用机会集。原始生产时钟与本轮延迟close时钟不同，不能把跨时钟收益差称作alpha。

历史仍为复用诊断，真实首收/成交/股息资格未取得；逐笔费用为当前费表投影，滑点为不利执行准备金。监测回撤使用15分钟报价，不是每秒保证。结论不晋升，目标保持active；本固定组合结束后不修改源模型、窗口、阈值或重选最佳特征组。

归因：首小时分钟数值在0/2/5bp三种情景均落后于同机会集的数值中性对照。改善原控制器的部分来自恢复权限/时段结构，不能认作分钟预测alpha。原生持仓默认近似清仓会丢失微小正余额，已复用QlibContinuousPosition，费用/价格不改、阈值不放宽；当前整段账户对账相对误差约4.4e-16。

只读Futu探测保存了盘前报价和本地接收时间，但未获得常规时段执行校准资格；原始探测仅在NAS Recorder保存，未下单。稳定超额目标保持active。


# 多时点完整Stage1费用检验

冻结分钟模型与预测复用，新增拟合0次；18个实际Tiered/0-2-5bp账户通过Qlib逐笔费用、现金、股数及净值对账。原首小时主候选保留，其他两组不事后重选。

|模型|滑点bp|净收益%|15分钟监测回撤%|相对SOXX财富|
|---|---:|---:|---:|---:|
|current|0|67.20|-30.38|0.3591|
|neutral_first_hour|0|201.15|-28.67|0.6467|
|first_hour|0|177.00|-31.82|0.5949|
|full_day|0|103.84|-33.59|0.4378|
|full_day_context|0|113.93|-32.07|0.4594|
|soxx|0|365.66|-43.40|1.0000|
|current|2|32.17|-32.31|0.2838|
|neutral_first_hour|2|228.11|-29.25|0.7046|
|first_hour|2|172.03|-32.51|0.5842|
|full_day|2|120.48|-32.37|0.4735|
|full_day_context|2|167.68|-30.58|0.5749|
|soxx|2|365.64|-43.41|1.0000|
|current|5|113.00|-29.13|0.4575|
|neutral_first_hour|5|196.31|-30.70|0.6364|
|first_hour|5|188.25|-30.81|0.6191|
|full_day|5|90.29|-32.33|0.4087|
|full_day_context|5|63.35|-33.23|0.3508|
|soxx|5|365.61|-43.41|1.0000|

门槛：{"net_soxx_excess_all_costs": false, "monitored_drawdown_within_30": false, "cvar_no_worse_than_soxx": true, "minute_increment_all_costs": false, "rolling_win_70pct": false, "positive_years_at_least_three_all_costs": false, "paired_bootstrap_excess_and_increment": false, "qualified_independent_forward": false}。原开盘动作、日级风险状态与恢复许可保留，盘中经济选择及中性对照共用机会集。原始生产时钟与本轮延迟close时钟不同，不能把跨时钟收益差称作alpha。

历史仍为复用诊断，真实首收/成交/股息资格未取得；逐笔费用为当前费表投影，滑点为不利执行准备金。监测回撤使用15分钟报价，不是每秒保证。结论不晋升，目标保持active；本固定组合结束后不修改源模型、窗口、阈值或重选最佳特征组。

归因：首小时分钟数值在0/2/5bp三种情景均落后于同机会集的数值中性对照。改善原控制器的部分来自恢复权限/时段结构，不能认作分钟预测alpha。原生持仓默认近似清仓会丢失微小正余额，已复用QlibContinuousPosition，费用/价格不改、阈值不放宽；当前整段账户对账相对误差约4.4e-16。

只读Futu探测保存了盘前报价和本地接收时间，但未获得常规时段执行校准资格；原始探测仅在NAS Recorder保存，未下单。稳定超额目标保持active。
